# Additional Baselines: Complexity Correction

In the following we demonstrate how to reproduce the complexity correction results.

We use the CIFAR-10 --> SVHN and CIFAR-10 --> CelebA setups as examples.

In [ ]:
import io
import pandas as pd
import torch
import numpy as np

from torchvision.transforms.functional import to_pil_image
from tqdm import tqdm

from sitn.datasets import create_dataset
from sitn.metrics import bootstrap_auroc
from sitn.utils import construct_results_path

## Compute PNG Complexity

In [ ]:
def png_complexity(img):
    """Returns PNG complexity in bits per dimension (bpd)."""
    buf = io.BytesIO()
    to_pil_image(img).save(buf, format="PNG")
    return buf.tell() * 8 / img.numel()

In [ ]:
complexity_corrections = {}
for dataset_name in ["cifar10", "svhn", "celeba"]:
    # Create dataset without normalisation
    dataset = create_dataset(
        dataset_name=dataset_name,
        image_size=[3, 32, 32],
        normalize_mean=0,  # no normalization
        normalize_std=1,  # no normalization
        pick="test",
        scratch_root=False,
    )

    records = []
    for sample_index, (img, _) in enumerate(tqdm(dataset, desc=f"{dataset_name}")):
        # img: float32 [C, H, W] in [0, 1] — scale back to uint8 for lossless compression
        img = (img * 255).clamp(0, 255).to(torch.uint8)
        records.append(
            {
                "eval_dataset": dataset_name,
                "sample_index": sample_index,
                "png_complexity_bpd": png_complexity(img),
            }
        )

    complexity_corrections[dataset_name] = pd.DataFrame(records)


## Evaluate OOD Detection Performance

In [ ]:
# Configurations
# We assume the model has already be trained and evaluated with
# these configurations (follow the cross-dataset OOD detection
# notebook to see how).

train_cfg = {"dataset_name": "cifar10"}
eval_cfg_test = {"config": train_cfg, "split_pick": "test"}

eval_cfg_svhn = {"config": train_cfg, "eval_dataset_name": "svhn", "split_pick": "test"}
eval_cfg_celeba = {"config": train_cfg, "eval_dataset_name": "celeba", "split_pick": "test"}

In [ ]:
# Metric configurations
metrics = {
    "log_likelihood_png_correction": {"label": "Complexity (PNG)", "higher_is_ood": True},
}

# Load ID test predictions
id_preds = pd.read_csv(construct_results_path(**eval_cfg_test, result_type="predictions"))
id_preds["train_dataset"] = eval_cfg_test["config"]["dataset_name"]
id_preds["eval_dataset"] = eval_cfg_test["config"]["dataset_name"]

# Add complexity correction
id_preds = id_preds.merge(
    complexity_corrections[eval_cfg_test["config"]["dataset_name"]],
    on=["eval_dataset", "sample_index"],
    how="left",
)

id_preds["log_likelihood_bpd"] = id_preds["log_likelihood"] / (3 * 32 * 32 * np.log(2))  # convert to bpd
id_preds["log_likelihood_png_correction"] = -id_preds["log_likelihood_bpd"] - id_preds["png_complexity_bpd"]

results = []
for eval_cfg_ood in [eval_cfg_svhn, eval_cfg_celeba]:
    # Load OOD test predictions
    ood_preds = pd.read_csv(construct_results_path(**eval_cfg_ood, result_type="predictions"))
    ood_preds["train_dataset"] = eval_cfg_ood["config"]["dataset_name"]
    ood_preds["eval_dataset"] = eval_cfg_ood["eval_dataset_name"]

    # Add complexity correction
    ood_preds = ood_preds.merge(
        complexity_corrections[eval_cfg_ood["eval_dataset_name"]],
        on=["eval_dataset", "sample_index"],
        how="left",
    )

    ood_preds["log_likelihood_bpd"] = ood_preds["log_likelihood"] / (3 * 32 * 32 * np.log(2))  # convert to bpd
    ood_preds["log_likelihood_png_correction"] = -ood_preds["log_likelihood_bpd"] - ood_preds["png_complexity_bpd"]

    # Combine ID and OOD predictions
    preds = pd.concat([id_preds.copy(), ood_preds], ignore_index=True)

    # Compute AUROC with bootstrapped CIs for each method
    y_true = (preds["eval_dataset"] != preds["train_dataset"]).astype(int)
    for col, meta in metrics.items():
        scores = preds[col].copy()
        if not meta["higher_is_ood"]:
            scores = -scores

        auroc, ci_lo, ci_hi = bootstrap_auroc(y_true, scores)
        results.append(
            {
                "ood_dataset": eval_cfg_ood["eval_dataset_name"],
                "metric": meta["label"],
                "AUROC": auroc,
                "CI_lo": ci_lo,
                "CI_hi": ci_hi,
            }
        )

results = pd.DataFrame(results).set_index(["ood_dataset", "metric"])
results

,,AUROC,CI_lo,CI_hi
ood_dataset,metric,,,
svhn,Complexity (PNG),0.781947,0.776622,0.787306
celeba,Complexity (PNG),0.615313,0.608122,0.622878
